In [ ]:
import os
from time import time
import pandas as pd
import xgboost as xgb
from additional.config import TEST_SPLIT_PERCENTAGE, RANDOM_STATE, RANDOM_CV_SPLIT
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from additional.helper import expand_contractions, read_data
import matplotlib.pyplot as plt

# Initialize the label encoder to convert string labels into numerical format
label_encoder = LabelEncoder()

# Set up a cross-validation strategy that maintains the distribution of labels across folds
cv_strategy = StratifiedKFold(n_splits=RANDOM_CV_SPLIT, shuffle=True, random_state=RANDOM_STATE)

# Load and preprocess the dataset
df = read_data()
df['label'] = label_encoder.fit_transform(df['punctuation'])

In [ ]:
# Train/Test split
df_train, df_test = train_test_split(df, test_size=TEST_SPLIT_PERCENTAGE, stratify=df['label'], random_state=RANDOM_STATE)

### Two cleaning preprocess classes:
1. **SimpleCleaningTransformer** - A simple transformer (lowercase and character removal).
2. **StopWordsRemovalTransformer** - In addition to simple cleaning, there is stop words removal.

In [ ]:
class SimpleCleaningTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Ensure X is a pandas Series
        if not isinstance(X, pd.Series):
            raise ValueError(f"Input should be a pandas Series.{type(X)}, {X.shape}")
        
        # Apply cleaning operations
        X_cleaned = X.str.lower()  # Convert to lowercase
        X_cleaned = X_cleaned.str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)  # Remove punctuation but keep numbers
        X_cleaned = X_cleaned.str.strip()  # Remove leading/trailing whitespace

        X_no_empty = [ EMPTY_TOKEN if row == '' else row for row in X_cleaned]
        
        return X_no_empty

class StopWordsRemovalTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Ensure X is a pandas Series
        if not isinstance(X, pd.Series):
            raise ValueError(f"Input should be a pandas Series.{type(X)}, {X.shape}")

        # Apply cleaning operations
        X_cleaned = X.str.lower()  # Convert to lowercase
        X_cleaned = X_cleaned.str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)  # Remove punctuation but keep numbers
        X_cleaned = X_cleaned.str.strip()  # Remove leading/trailing whitespace

        X_no_stop_words = [' '.join([word for word in row.split() if word not in ENGLISH_STOP_WORDS]) for row in X_cleaned]
        X_no_stop_words_no_empty = [ EMPTY_TOKEN if row == '' else row for row in X_no_stop_words]

        return X_no_stop_words_no_empty

In [ ]:
# vocabulary_dict = None
pipeline = Pipeline(steps=[("cleaning","tmp_cleaner"),
                           ("bow", CountVectorizer()),
                           ("tfidf", TfidfTransformer()),
                           ('classifier', AdaBoostClassifier())])

# Bootstrap aggregating
RandomForestClassifier_param = {'cleaning': [SimpleCleaningTransformer(), StopWordsRemovalTransformer()],
                                'classifier': [RandomForestClassifier()],
                                'classifier__n_estimators': [100],
                                'classifier__max_depth': [10],
                                'classifier__class_weight': ['balanced'],
                                'classifier__min_samples_split': [2],
                                'classifier__random_state': [RANDOM_STATE]}
# AdaBoost
AdaBoost_param = {'cleaning': [SimpleCleaningTransformer(), StopWordsRemovalTransformer()],
                  'classifier': [AdaBoostClassifier(algorithm='SAMME')], #DecisionTree
                  'classifier__n_estimators': [100, 200],
                  'classifier__learning_rate': [0.1, 1],
                  'classifier__random_state': [RANDOM_STATE]}

# XGBoost
XGBClassifier_param = {'cleaning': [SimpleCleaningTransformer(), StopWordsRemovalTransformer()],
                       'classifier': [xgb.XGBClassifier()],
                       'classifier__eta': [0.1],
                       'classifier__gamma': [1.5],
                       'classifier__max_depth': [6],
                       'classifier__subsample': [1.0],
                       'classifier__min_child_weight': [1.0],
                       'classifier__eval_metric': ['mlogloss'],
                       'classifier__seed': [RANDOM_STATE]}

params = [AdaBoost_param, XGBClassifier_param, RandomForestClassifier_param]

gs = GridSearchCV(pipeline, param_grid=params, refit='f1', return_train_score=True, cv=cv_strategy, n_jobs=-1, scoring='f1_micro')

In [ ]:
tic = time()
print('df_train shape:',df_train.shape)
gs.fit(df_train['t_uncontraction'], df_train['label'])
toc = time()
print(f'GridSearchCV time(mintues): {round((toc-tic)/60,2)}')

best_params = gs.best_params_

In [ ]:
pd.DataFrame(gs.cv_results_)

In [ ]:
best_model = gs.best_estimator_
best_model

In [ ]:
df_test['predictions'] = best_model.predict(df_test['t_uncontraction'])
f1_macro =  f1_score(df_test['label'], df_test['predictions'], average='macro')

print(f'F1 Test Score: {f1_macro}')

### check each label, F1 score

In [ ]:
labels = df_test['punctuation'].unique()

f1_scores = []
x_labels = []

for label in labels:
    label_number = label_encoder.transform([label])[0]
    f1 = f1_score(df_test['punctuation'] == label, df_test['predictions'] == label_number)
    f1_scores.append(f1)
    x_labels.append(str(label) + ' (Counts:' + str(sum(df_test['label'] == label_number)) +')')

plt.figure(figsize=(8, 5))
plt.bar(x_labels, f1_scores, color='skyblue', alpha=0.7)

plt.ylim(0, 1) 
plt.ylabel('F1 Score')
plt.title('F1 Score for Each Label')

# Show the plot
plt.show()

# Discussion and conclusion

1. Since there is F1=0 for '?' the max F1 Macro that we can get is 66%.
2. Since there are only 18 samples in the test set for '?' and the score is zero. We need to rethink about the the equal relevance for each category.
3. Instead of F1-macro on the imbalanced data we can use SMOTE and balance the data set.
4. Instead of simple tf-idf we can use embedding to get better prediction.
5. The test score of F1 = 0.38 is a good baseline for further investigation.
6. The next step, we can build a model that is fed two sentences and it tries to predict the punctuation between them.